# vLLM Inference Benchmark

## Objective

Find the maximum request rate (QPS) each server configuration can sustain
while meeting a real latency SLO, and identify the best `max-num-seqs` /
parallelism configuration for that goal.

### Independent variables

- `max-num-seqs`: 16, 32, 64, 96, 160
- `request-rate` (QPS): 0.1, 0.2, 0.3, 0.5, 1, 2
- Parallelism config: 1x GPU baseline, tensor-parallel (2 GPU), pipeline-parallel (2 GPU)

### Fixed variables

- Model: Qwen/Qwen2.5-1.5B-Instruct
- GPU: 2x NVIDIA Tesla T4
- max-model-len: 2000
- gpu-memory-utilization: 0.85
- Input length: 430 tokens
- Output length: 1000 tokens
- Requests per run: 50
- Repetitions per configuration: 3 (plus 1 warm-up, discarded)

### Service-level objective (SLO)

- p99 TTFT < 2000 ms
- p99 TPOT < 50 ms

A configuration's **goodput** is its output throughput counted only for
runs that met both SLO thresholds. The **max sustainable QPS** for a
given `max-num-seqs` / parallelism pair is the highest request rate at
which the SLO still holds.

## 1. Imports

In [ ]:
import os
import re
import sys
import json
import time
import signal
import subprocess
import threading
import itertools
from pathlib import Path
from datetime import datetime, timezone

import yaml
import requests
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version)

## 2. Load Configuration (single source of truth)

In [ ]:
# Configuration lives in configs/baseline.yaml and workloads/workload.yaml
# so the notebook, shell scripts, and README all read the same values.
# If the repo layout isn't found (e.g. running standalone on Kaggle),
# fall back to the defaults below.

_CANDIDATE_ROOTS = [Path(".."), Path("."), Path("/kaggle/working/llm-inference-benchmark")]

def _find_repo_root():
    for root in _CANDIDATE_ROOTS:
        if (root / "configs" / "baseline.yaml").exists():
            return root
    return None

REPO_ROOT = _find_repo_root()

_DEFAULT_BASELINE = {
    "model": {"name": "Qwen/Qwen2.5-1.5B-Instruct", "served_name": "llm-model"},
    "vllm": {"version": "0.30.0", "max_model_len": 2000, "gpu_memory_utilization": 0.85},
    "parallelism_configs": [
        {"name": "baseline_1gpu", "tensor_parallel_size": 1, "pipeline_parallel_size": 1, "gpus": [0]},
        {"name": "tensor_parallel_2gpu", "tensor_parallel_size": 2, "pipeline_parallel_size": 1, "gpus": [0, 1]},
        {"name": "pipeline_parallel_2gpu", "tensor_parallel_size": 1, "pipeline_parallel_size": 2, "gpus": [0, 1]},
    ],
    "workload": {"dataset": "random", "input_len": 430, "output_len": 1000, "num_prompts": 50},
    "experiment": {
        "max_num_seqs": [16, 32, 64, 96, 160],
        "request_rate": [0.1, 0.2, 0.3, 0.5, 1, 2],
        "repetitions": 3,
        "warmup": True,
    },
    "slo": {"ttft_ms": 2000, "tpot_ms": 50},
    "cost": {"hourly_gpu_cost_usd": 0.526},
}

if REPO_ROOT is not None:
    with open(REPO_ROOT / "configs" / "baseline.yaml") as f:
        baseline_cfg = yaml.safe_load(f)
    print("Loaded config from:", REPO_ROOT / "configs" / "baseline.yaml")
else:
    baseline_cfg = _DEFAULT_BASELINE
    print("configs/baseline.yaml not found — using built-in defaults.")

MODEL = baseline_cfg["model"]["name"]
SERVED_NAME = baseline_cfg["model"]["served_name"]

HOST = "127.0.0.1"
PORT = 8000
BASE_URL = f"http://{HOST}:{PORT}"

MAX_MODEL_LEN = baseline_cfg["vllm"]["max_model_len"]
GPU_MEMORY_UTILIZATION = baseline_cfg["vllm"]["gpu_memory_utilization"]
EXPECTED_VLLM_VERSION = baseline_cfg["vllm"]["version"]

PARALLELISM_CONFIGS = baseline_cfg["parallelism_configs"]

MAX_NUM_SEQS_VALUES = baseline_cfg["experiment"]["max_num_seqs"]
REQUEST_RATE_VALUES = baseline_cfg["experiment"]["request_rate"]
REPETITIONS = baseline_cfg["experiment"]["repetitions"]
WARMUP_ENABLED = baseline_cfg["experiment"]["warmup"]

NUM_PROMPTS = baseline_cfg["workload"]["num_prompts"]
INPUT_LEN = baseline_cfg["workload"]["input_len"]
OUTPUT_LEN = baseline_cfg["workload"]["output_len"]
DATASET = baseline_cfg["workload"]["dataset"]

SLO_TTFT_MS = baseline_cfg["slo"]["ttft_ms"]
SLO_TPOT_MS = baseline_cfg["slo"]["tpot_ms"]
HOURLY_GPU_COST_USD = baseline_cfg["cost"]["hourly_gpu_cost_usd"]

RESULT_DIR = Path("results/raw")
PLOT_DIR = Path("plots")
LOG_DIR = Path("logs")
CHECKPOINT_PATH = Path("checkpoint.json")
METADATA_PATH = RESULT_DIR / "run-metadata.jsonl"

for d in (RESULT_DIR, PLOT_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

TOTAL_RUNS_PER_SERVER = len(REQUEST_RATE_VALUES) * (REPETITIONS + int(WARMUP_ENABLED))
TOTAL_SERVERS = len(PARALLELISM_CONFIGS) * len(MAX_NUM_SEQS_VALUES)
TOTAL_RUNS = TOTAL_SERVERS * TOTAL_RUNS_PER_SERVER

print("Model:", MODEL)
print("Parallelism configs:", [p["name"] for p in PARALLELISM_CONFIGS])
print("max-num-seqs values:", MAX_NUM_SEQS_VALUES)
print("request-rate values:", REQUEST_RATE_VALUES)
print("Repetitions:", REPETITIONS, "| Warmup:", WARMUP_ENABLED)
print("SLO: p99 TTFT <", SLO_TTFT_MS, "ms, p99 TPOT <", SLO_TPOT_MS, "ms")
print()
print(f"Planned server starts: {TOTAL_SERVERS}")
print(f"Planned benchmark invocations: {TOTAL_RUNS}")
print("Review this before running the full sweep (Section 10) — reduce")
print("REQUEST_RATE_VALUES or REPETITIONS in configs/baseline.yaml if this")
print("won't fit your Kaggle session window.")

## 3. GPU Verification

In [ ]:
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("nvidia-smi failed")

gpu_count = subprocess.run(
    ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip().splitlines()

print(f"Detected {len(gpu_count)} GPU(s): {gpu_count}")

max_gpus_needed = max(len(p["gpus"]) for p in PARALLELISM_CONFIGS)
if len(gpu_count) < max_gpus_needed:
    print(f"WARNING: a parallelism config needs {max_gpus_needed} GPU(s) "
          f"but only {len(gpu_count)} detected.")

## 4. vLLM Verification

In [ ]:
import importlib.util

if importlib.util.find_spec("vllm") is None:
    print(f"vLLM not found. Installing vLLM {EXPECTED_VLLM_VERSION}...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", f"vllm=={EXPECTED_VLLM_VERSION}"],
        check=True
    )
else:
    print("vLLM package already installed.")

In [ ]:
import vllm

print("vLLM version:", vllm.__version__)

if vllm.__version__ != EXPECTED_VLLM_VERSION:
    print(f"WARNING: expected vLLM {EXPECTED_VLLM_VERSION}, found {vllm.__version__}")

## 5. GPU Utilization Monitor

Samples `nvidia-smi` in the background while a server is running so GPU
utilization and memory pressure are captured per configuration, not just
inferred after the fact.

In [ ]:
class GpuMonitor:
    def __init__(self, log_path, interval_s=1.0):
        self.log_path = Path(log_path)
        self.interval_s = interval_s
        self._stop_event = threading.Event()
        self._thread = None

    def _run(self):
        with open(self.log_path, "w") as f:
            f.write("timestamp,gpu_index,utilization_pct,memory_used_mib,memory_total_mib\n")
            while not self._stop_event.is_set():
                result = subprocess.run(
                    ["nvidia-smi",
                     "--query-gpu=index,utilization.gpu,memory.used,memory.total",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True
                )
                ts = datetime.now(timezone.utc).isoformat()
                for line in result.stdout.strip().splitlines():
                    parts = [p.strip() for p in line.split(",")]
                    if len(parts) == 4:
                        f.write(f"{ts},{parts[0]},{parts[1]},{parts[2]},{parts[3]}\n")
                f.flush()
                self._stop_event.wait(self.interval_s)

    def start(self):
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=5)

## 6. Checkpoint / Resume

Kaggle sessions can disconnect mid-sweep. Progress is written to
`checkpoint.json` after every completed run so a restarted session can
skip work already done instead of starting over.

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH) as f:
            return set(json.load(f))
    return set()

def save_checkpoint(done_keys):
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(sorted(done_keys), f, indent=2)

def run_key(parallelism_name, max_num_seqs, request_rate, run_id):
    return f"{parallelism_name}|{max_num_seqs}|{request_rate}|{run_id}"

completed_runs = load_checkpoint()
print(f"{len(completed_runs)} run(s) already completed (from checkpoint.json).")

## 7. Server Lifecycle

In [ ]:
server_process = None

def start_vllm(parallelism, max_num_seqs):
    global server_process

    if server_process is not None and server_process.poll() is None:
        raise RuntimeError("vLLM server is already running.")

    gpus = parallelism["gpus"]
    tp = parallelism["tensor_parallel_size"]
    pp = parallelism["pipeline_parallel_size"]

    if tp * pp != len(gpus):
        raise ValueError(
            f"parallelism config '{parallelism['name']}': "
            f"tp({tp}) * pp({pp}) != len(gpus)({len(gpus)})"
        )

    command = [
        "vllm", "serve", MODEL,
        "--served-model-name", SERVED_NAME,
        "--host", "0.0.0.0",
        "--port", str(PORT),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--max-num-seqs", str(max_num_seqs),
        "--tensor-parallel-size", str(tp),
        "--pipeline-parallel-size", str(pp),
        "--dtype", "half",
    ]

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in gpus)

    log_path = LOG_DIR / f"vllm-{parallelism['name']}-seqs{max_num_seqs}.log"
    log_file = open(log_path, "w")

    print("Starting vLLM:")
    print(" ".join(command))
    print("CUDA_VISIBLE_DEVICES:", env["CUDA_VISIBLE_DEVICES"])
    print("Log:", log_path)

    server_process = subprocess.Popen(
        command, stdout=log_file, stderr=subprocess.STDOUT, env=env,
    )

    return server_process

In [ ]:
def wait_for_server(timeout=300):
    """Blocks until the server is healthy. Returns elapsed boot time in seconds."""
    health_url = f"{BASE_URL}/health"
    start_time = time.time()

    while time.time() - start_time < timeout:
        if server_process is not None and server_process.poll() is not None:
            raise RuntimeError(
                "vLLM server exited before becoming healthy. Check its log file."
            )
        try:
            response = requests.get(health_url, timeout=5)
            if response.status_code == 200:
                elapsed = time.time() - start_time
                print(f"vLLM server is healthy after {elapsed:.1f}s.")
                return elapsed
        except requests.RequestException:
            pass
        time.sleep(2)

    raise TimeoutError(f"vLLM server did not become healthy within {timeout} seconds.")

In [ ]:
def get_model_info():
    response = requests.get(f"{BASE_URL}/v1/models", timeout=10)
    response.raise_for_status()
    data = response.json()
    print(json.dumps(data, indent=2))
    return data

In [ ]:
def stop_vllm():
    global server_process

    if server_process is None:
        print("No server process.")
        return

    if server_process.poll() is None:
        print("Stopping vLLM server...")
        server_process.terminate()
        try:
            server_process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            print("Server did not stop gracefully. Killing...")
            server_process.kill()
            server_process.wait()
    else:
        print("vLLM server already stopped.")

    server_process = None

## 8. Benchmark Runner

In [ ]:
def run_benchmark(max_num_seqs, request_rate, run_id):
    rate_label = str(request_rate).replace(".", "p")
    result_filename = f"max-num-seqs-{max_num_seqs}-rate-{rate_label}-run-{run_id}.json"
    result_path = RESULT_DIR / result_filename

    command = [
        "vllm", "bench", "serve",
        "--backend", "vllm",
        "--base-url", BASE_URL,
        "--model", MODEL,
        "--served-model-name", SERVED_NAME,
        "--dataset-name", DATASET,
        "--random-input-len", str(INPUT_LEN),
        "--random-output-len", str(OUTPUT_LEN),
        "--num-prompts", str(NUM_PROMPTS),
        "--request-rate", str(request_rate),
        "--save-result",
        "--result-dir", str(RESULT_DIR),
        "--result-filename", result_filename,
    ]

    print("=" * 70)
    print(f"Running benchmark: max-num-seqs={max_num_seqs}, "
          f"request-rate={request_rate}, run={run_id}")
    print("=" * 70)
    print(" ".join(command))

    result = subprocess.run(command, text=True)

    if result.returncode != 0:
        raise RuntimeError(
            f"Benchmark failed: max-num-seqs={max_num_seqs}, "
            f"request-rate={request_rate}, run={run_id}"
        )
    if not result_path.exists():
        raise FileNotFoundError(f"Expected result file not found: {result_path}")

    print("Result saved:", result_path)
    return result_path

## 9. Smoke Test

Starts the server once at a small `max-num-seqs`, confirms it answers a
completion request, then stops it. Confirms the pipeline works before
committing to the full sweep.

In [ ]:
smoke_parallelism = PARALLELISM_CONFIGS[0]
smoke_max_num_seqs = MAX_NUM_SEQS_VALUES[0]

start_vllm(smoke_parallelism, smoke_max_num_seqs)

try:
    wait_for_server()
    get_model_info()

    payload = {
        "model": SERVED_NAME,
        "prompt": "Explain Kubernetes GPU scheduling in simple terms.",
        "max_tokens": 20,
        "temperature": 0,
    }
    response = requests.post(f"{BASE_URL}/v1/completions", json=payload, timeout=120)
    print("HTTP status:", response.status_code)
    response.raise_for_status()
    print(json.dumps(response.json(), indent=2)[:2000])

finally:
    stop_vllm()

## 10. Full Sweep: Parallelism x max-num-seqs x request-rate

Starts one server per (parallelism, max-num-seqs) pair, then runs every
request-rate value (warm-up + repetitions) against that same server
before moving on. A failure in one configuration is logged and skipped
rather than aborting the rest of the sweep. Progress is checkpointed
after every run.

In [ ]:
run_metadata_records = []

for parallelism in PARALLELISM_CONFIGS:
    for max_num_seqs in MAX_NUM_SEQS_VALUES:

        print()
        print("#" * 80)
        print(f"# CONFIG: parallelism={parallelism['name']}, max-num-seqs={max_num_seqs}")
        print("#" * 80)

        pending_keys = [
            run_key(parallelism["name"], max_num_seqs, rate, run_id)
            for rate in REQUEST_RATE_VALUES
            for run_id in (["warmup"] if WARMUP_ENABLED else []) + list(range(1, REPETITIONS + 1))
        ]
        if all(k in completed_runs for k in pending_keys):
            print("All runs for this configuration already completed. Skipping.")
            continue

        try:
            start_vllm(parallelism, max_num_seqs)
            cold_start_s = wait_for_server()
            get_model_info()
        except Exception as e:
            print(f"FAILED to start server for {parallelism['name']} / "
                  f"max-num-seqs={max_num_seqs}: {e}")
            stop_vllm()
            time.sleep(5)
            continue

        monitor = GpuMonitor(
            LOG_DIR / f"gpu-util-{parallelism['name']}-seqs{max_num_seqs}.csv"
        )
        monitor.start()

        try:
            for request_rate in REQUEST_RATE_VALUES:
                run_ids = (["warmup"] if WARMUP_ENABLED else []) + list(range(1, REPETITIONS + 1))

                for run_id in run_ids:
                    key = run_key(parallelism["name"], max_num_seqs, request_rate, run_id)
                    if key in completed_runs:
                        print(f"Already done, skipping: {key}")
                        continue

                    try:
                        result_path = run_benchmark(max_num_seqs, request_rate, run_id)

                        run_metadata_records.append({
                            "result_file": str(result_path),
                            "parallelism": parallelism["name"],
                            "tensor_parallel_size": parallelism["tensor_parallel_size"],
                            "pipeline_parallel_size": parallelism["pipeline_parallel_size"],
                            "gpus": parallelism["gpus"],
                            "max_num_seqs": max_num_seqs,
                            "request_rate": request_rate,
                            "run_id": run_id,
                            "cold_start_s": cold_start_s,
                            "vllm_version": vllm.__version__,
                            "timestamp": datetime.now(timezone.utc).isoformat(),
                        })
                        with open(METADATA_PATH, "a") as f:
                            f.write(json.dumps(run_metadata_records[-1]) + "\n")

                        completed_runs.add(key)
                        save_checkpoint(completed_runs)

                    except Exception as e:
                        print(f"FAILED run {key}: {e}")
                        continue

        finally:
            monitor.stop()
            stop_vllm()

        time.sleep(5)

print()
print("Sweep complete.")
print(f"{len(completed_runs)} run(s) recorded in checkpoint.json.")

## 11. Inspect Raw Results

In [ ]:
result_files = sorted(RESULT_DIR.glob("max-num-seqs-*.json"))
print(f"{len(result_files)} raw result file(s) found.")
for path in result_files[:5]:
    print(" -", path)
if len(result_files) > 5:
    print(f"   ... and {len(result_files) - 5} more")

## 12. Parse Benchmark Results

In [ ]:
FILENAME_RE = re.compile(
    r"max-num-seqs-(?P<max_num_seqs>\d+)-rate-(?P<rate>[0-9p]+)-run-(?P<run_id>\w+)\.json"
)

def find_metric(data, possible_names):
    for name in possible_names:
        if name in data:
            return data[name]
    return None

def parse_result(path):
    with open(path) as f:
        data = json.load(f)

    match = FILENAME_RE.match(path.name)
    if not match:
        raise ValueError(f"Filename does not match expected pattern: {path.name}")

    row = {
        "max_num_seqs": int(match.group("max_num_seqs")),
        "request_rate": float(match.group("rate").replace("p", ".")),
        "run_id": match.group("run_id"),
        "file": str(path),
    }

    row["request_throughput"] = find_metric(data, ["request_throughput"])
    row["output_throughput"] = find_metric(data, ["output_throughput"])
    row["total_throughput"] = find_metric(data, ["total_token_throughput", "total_throughput"])

    row["mean_ttft_ms"] = find_metric(data, ["mean_ttft_ms"])
    row["p99_ttft_ms"] = find_metric(data, ["p99_ttft_ms"])
    row["mean_tpot_ms"] = find_metric(data, ["mean_tpot_ms"])
    row["p99_tpot_ms"] = find_metric(data, ["p99_tpot_ms"])

    # Not all vLLM versions report queue time as a separate field —
    # this stays None if the field isn't present rather than guessing.
    row["mean_queue_time_ms"] = find_metric(
        data, ["mean_queue_time_ms", "mean_waiting_time_ms"]
    )

    row["completed"] = find_metric(data, ["completed"])
    row["failed"] = find_metric(data, ["failed"])

    return row

rows = []
for path in result_files:
    try:
        rows.append(parse_result(path))
    except Exception as e:
        print("Could not parse:", path, "-", e)

results_df = pd.DataFrame(rows)

# Attach run metadata (parallelism config, cold start, vllm version) by filename.
if METADATA_PATH.exists():
    meta_rows = [json.loads(line) for line in open(METADATA_PATH)]
    meta_df = pd.DataFrame(meta_rows).rename(columns={"result_file": "file"})
    results_df = results_df.merge(
        meta_df[["file", "parallelism", "cold_start_s", "vllm_version", "timestamp"]],
        on="file", how="left"
    )

results_df

## 13. Save Summary

In [ ]:
summary_path = RESULT_DIR / "benchmark-summary.csv"
results_df.to_csv(summary_path, index=False)
print("Saved:", summary_path)
results_df

## 14. Goodput and SLO Pass/Fail

Warm-up runs are excluded from analysis. Each remaining run is checked
against the SLO; **goodput** is output throughput counted only for runs
that met it.

In [ ]:
measured_df = results_df[results_df["run_id"] != "warmup"].copy()

measured_df["slo_pass"] = (
    (measured_df["p99_ttft_ms"] < SLO_TTFT_MS) &
    (measured_df["p99_tpot_ms"] < SLO_TPOT_MS)
)
measured_df["goodput"] = measured_df["output_throughput"].where(measured_df["slo_pass"], 0.0)

config_summary = (
    measured_df
    .groupby(["parallelism", "max_num_seqs", "request_rate"], as_index=False)
    .agg(
        p99_ttft_ms=("p99_ttft_ms", "mean"),
        p99_tpot_ms=("p99_tpot_ms", "mean"),
        output_throughput=("output_throughput", "mean"),
        goodput=("goodput", "mean"),
        slo_pass=("slo_pass", "all"),
        failed=("failed", "sum"),
    )
)

config_summary

## 15. Plot: Output Throughput vs Request Rate

In [ ]:
for parallelism_name, group in config_summary.groupby("parallelism"):
    plt.figure(figsize=(8, 5))
    for mns, sub in group.groupby("max_num_seqs"):
        sub = sub.sort_values("request_rate")
        plt.plot(sub["request_rate"], sub["output_throughput"], marker="o", label=f"max-num-seqs={mns}")
    plt.xlabel("Request rate (req/s, offered)")
    plt.ylabel("Output throughput (tokens/s)")
    plt.title(f"Output Throughput vs Request Rate — {parallelism_name}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = PLOT_DIR / f"throughput-vs-request-rate-{parallelism_name}.png"
    plt.savefig(out_path, dpi=150)
    plt.show()
    print("Saved:", out_path)

## 16. Plot: p99 TTFT vs Request Rate (SLO line)

In [ ]:
for parallelism_name, group in config_summary.groupby("parallelism"):
    plt.figure(figsize=(8, 5))
    for mns, sub in group.groupby("max_num_seqs"):
        sub = sub.sort_values("request_rate")
        plt.plot(sub["request_rate"], sub["p99_ttft_ms"], marker="o", label=f"max-num-seqs={mns}")
    plt.axhline(SLO_TTFT_MS, color="red", linestyle="--", label=f"SLO ({SLO_TTFT_MS} ms)")
    plt.xlabel("Request rate (req/s, offered)")
    plt.ylabel("p99 TTFT (ms)")
    plt.title(f"p99 TTFT vs Request Rate — {parallelism_name}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = PLOT_DIR / f"p99-ttft-vs-request-rate-{parallelism_name}.png"
    plt.savefig(out_path, dpi=150)
    plt.show()
    print("Saved:", out_path)

## 17. Plot: p99 TPOT vs Request Rate (SLO line)

In [ ]:
for parallelism_name, group in config_summary.groupby("parallelism"):
    plt.figure(figsize=(8, 5))
    for mns, sub in group.groupby("max_num_seqs"):
        sub = sub.sort_values("request_rate")
        plt.plot(sub["request_rate"], sub["p99_tpot_ms"], marker="o", label=f"max-num-seqs={mns}")
    plt.axhline(SLO_TPOT_MS, color="red", linestyle="--", label=f"SLO ({SLO_TPOT_MS} ms)")
    plt.xlabel("Request rate (req/s, offered)")
    plt.ylabel("p99 TPOT (ms)")
    plt.title(f"p99 TPOT vs Request Rate — {parallelism_name}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = PLOT_DIR / f"p99-tpot-vs-request-rate-{parallelism_name}.png"
    plt.savefig(out_path, dpi=150)
    plt.show()
    print("Saved:", out_path)

## 18. Max Sustainable QPS per Configuration

For each (parallelism, max-num-seqs) pair, the highest request rate that
still meets the SLO. This is the number to use when deciding capacity,
not raw max throughput.

In [ ]:
def max_sustainable_rate(group):
    passing = group[group["slo_pass"]]
    if passing.empty:
        return pd.Series({"max_sustainable_qps": 0.0, "goodput_at_max_qps": 0.0})
    best = passing.loc[passing["request_rate"].idxmax()]
    return pd.Series({
        "max_sustainable_qps": best["request_rate"],
        "goodput_at_max_qps": best["goodput"],
    })

sustainable = (
    config_summary
    .groupby(["parallelism", "max_num_seqs"])
    .apply(max_sustainable_rate)
    .reset_index()
)

sustainable.sort_values("goodput_at_max_qps", ascending=False)

## 19. Cost per 1M Output Tokens (at max sustainable QPS)

Cost is computed on **goodput**, not raw throughput, and scaled by how
many GPUs each parallelism config actually uses. Replace
`hourly_gpu_cost_usd` in `configs/baseline.yaml` with your real provider
rate before trusting these numbers.

In [ ]:
gpus_used_by_config = {p["name"]: len(p["gpus"]) for p in PARALLELISM_CONFIGS}

cost_df = sustainable.copy()
cost_df["gpus_used"] = cost_df["parallelism"].map(gpus_used_by_config)
cost_df["hourly_cost_usd"] = cost_df["gpus_used"] * HOURLY_GPU_COST_USD
cost_df["tokens_per_hour"] = cost_df["goodput_at_max_qps"] * 3600

cost_df["cost_per_1m_output_tokens_usd"] = None
has_tokens = cost_df["tokens_per_hour"] > 0
cost_df.loc[has_tokens, "cost_per_1m_output_tokens_usd"] = (
    cost_df.loc[has_tokens, "hourly_cost_usd"]
    / cost_df.loc[has_tokens, "tokens_per_hour"]
    * 1_000_000
)

cost_df.sort_values("cost_per_1m_output_tokens_usd")

## 20. Package Artifacts

In [ ]:
from IPython.display import FileLink, display

artifact_path = Path("benchmark-artifacts.zip")
subprocess.run(
    ["zip", "-r", str(artifact_path), str(LOG_DIR), str(RESULT_DIR), str(PLOT_DIR),
     "checkpoint.json"],
    check=True
)

print("Artifact size:")
subprocess.run(["ls", "-lh", str(artifact_path)])

display(FileLink(str(artifact_path)))

## 21. Benchmark Summary

This experiment sweeps three variables together — `max-num-seqs`,
offered `request-rate`, and GPU parallelism config — instead of holding
request rate fixed. That's what makes it possible to find each
configuration's actual **max sustainable QPS** under a real SLO, rather
than only observing overloaded-queue behavior at one fixed rate.

Read `sustainable` (Section 18) and `cost_df` (Section 19) for the
final answer: which `max-num-seqs` / parallelism combination serves the
most traffic within the SLO, and at what cost per token.